# v84 -- Pierre Auger UHECR Hemispheric Asymmetry (CORRECTED)
## KTF³-R axis test: l=277°, b=-31°
**Author:** Andrew Cotting | 5 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

Data: Auger Open Data summary.zip | DOI: 10.5281/zenodo.4487612
Fix: sd_l/sd_b are local coords — using sd_ra + zenith→dec conversion
Method: RA scrambling for exposure correction (standard Auger method)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd, zipfile

print('='*60)
print('v84 -- Auger UHECR (CORRECTED coords + RA scrambling)')
print('='*60)

def gal_vec(l_deg, b_deg):
    l, b = np.radians(l_deg), np.radians(b_deg)
    return np.array([np.cos(b)*np.cos(l), np.cos(b)*np.sin(l), np.sin(b)])

def radec_to_gal(ra_deg, dec_deg):
    ra_r = np.radians(ra_deg); dec_r = np.radians(dec_deg)
    ra_NGP = np.radians(192.85948); dec_NGP = np.radians(27.12825)
    sin_b = (np.sin(dec_r)*np.sin(dec_NGP) +
             np.cos(dec_r)*np.cos(dec_NGP)*np.cos(ra_r - ra_NGP))
    b = np.degrees(np.arcsin(np.clip(sin_b, -1, 1)))
    x = np.cos(dec_r)*np.sin(ra_r - ra_NGP)
    y = (np.sin(dec_r)*np.cos(dec_NGP) -
         np.cos(dec_r)*np.sin(dec_NGP)*np.cos(ra_r - ra_NGP))
    l = (122.93192 - np.degrees(np.arctan2(x, y))) % 360
    return l, b

axis_ktf3   = gal_vec(277.0, -31.0)
axis_dipole = gal_vec(233.0, -13.0)

# Load all Auger data
dfs = []
with zipfile.ZipFile('summary.zip', 'r') as z:
    for name in z.namelist():
        if name.endswith('.csv'):
            with z.open(name) as f:
                dfs.append(pd.read_csv(f))
            print(f'{name}: {len(dfs[-1]):,}')
df = pd.concat(dfs, ignore_index=True)

# Use sd_ra (RA) and sd_theta (zenith angle) to get Dec
# Auger lat = -35.2°. Dec = lat_obs + (90° - zenith) for transit
# Better: use sd_ra directly, compute dec from zenith + Auger latitude
# Dec ≈ arcsin(sin(lat)*cos(zenith) + cos(lat)*sin(zenith)*cos(azimuth))
# But simpler: Auger provides sd_ra. For dec, use:
# The event dec can be inferred, but sd_ra is the key column.
# For RA scrambling we only need RA and Dec.
# Compute Dec from Zenith + Auger geometry:
lat_auger = np.radians(-35.2)  # Mendoza, Argentina
zenith_r = np.radians(df['sd_theta'].values)
phi_r = np.radians(df['sd_phi'].values)  # azimuth from North
# Dec = arcsin(sin(lat)*cos(zen) + cos(lat)*sin(zen)*cos(phi))
sin_dec = (np.sin(lat_auger)*np.cos(zenith_r) +
           np.cos(lat_auger)*np.sin(zenith_r)*np.cos(phi_r))
dec_ev = np.degrees(np.arcsin(np.clip(sin_dec, -1, 1)))
ra_ev  = df['sd_ra'].values
e_ev   = df['sd_energy'].values

mask = np.isfinite(ra_ev) & np.isfinite(dec_ev) & np.isfinite(e_ev)
ra_ev, dec_ev, e_ev = ra_ev[mask], dec_ev[mask], e_ev[mask]
print(f'Valid: {mask.sum():,}')
print(f'RA:  {ra_ev.min():.1f} - {ra_ev.max():.1f}')
print(f'Dec: {dec_ev.min():.1f} - {dec_ev.max():.1f}')

# Verify: dipole should be ~3sigma at E>8 EeV
l_ev, b_ev2 = radec_to_gal(ra_ev, dec_ev)
l_r = np.radians(l_ev); b_r = np.radians(b_ev2)
vecs = np.column_stack([np.cos(b_r)*np.cos(l_r),
                        np.cos(b_r)*np.sin(l_r),
                        np.sin(b_r)])

# Check dipole direction
dot_d = vecs @ axis_dipole
dot_k = vecs @ axis_ktf3
sep = np.degrees(np.arccos(np.clip(axis_ktf3 @ axis_dipole, -1, 1)))
print(f'KTF³ vs Auger dipole: {sep:.1f}° (should be 22.9°)')


In [ ]:
# RA SCRAMBLING
def hemi_asym(vecs, axis):
    d = vecs @ axis
    nn, ns = np.sum(d>0), np.sum(d<0)
    return (nn-ns)/(nn+ns), d

obs_k, dot_k = hemi_asym(vecs, axis_ktf3)
obs_d, dot_d = hemi_asym(vecs, axis_dipole)

print('RA scrambling (500 iter)...')
np.random.seed(277)
sc_k, sc_d = [], []
for _ in range(500):
    ra_s = np.random.uniform(0, 360, len(ra_ev))
    ls, bs = radec_to_gal(ra_s, dec_ev)
    lrs = np.radians(ls); brs = np.radians(bs)
    vs = np.column_stack([np.cos(brs)*np.cos(lrs),
                          np.cos(brs)*np.sin(lrs),
                          np.sin(brs)])
    ak, _ = hemi_asym(vs, axis_ktf3)
    ad, _ = hemi_asym(vs, axis_dipole)
    sc_k.append(ak); sc_d.append(ad)
sc_k, sc_d = np.array(sc_k), np.array(sc_d)

sigma_k = (obs_k - sc_k.mean()) / sc_k.std()
sigma_d = (obs_d - sc_d.mean()) / sc_d.std()
p_val   = np.mean(np.abs(sc_k-sc_k.mean()) >= np.abs(obs_k-sc_k.mean()))
verdict = 'SIGNAL' if abs(sigma_k)>2 else ('HINT' if abs(sigma_k)>1 else 'NULL')

print(f'\nRESULTS (exposure-corrected):')
print(f'  KTF³  sigma = {sigma_k:+.3f}')
print(f'  Dipole sigma = {sigma_d:+.3f}  (expected ~3 at E>8 EeV)')
print(f'  Separation = {sep:.1f}°  (expected 22.9°)')
print(f'  p-value = {p_val:.3f}')
print(f'  VERDICT: {verdict}')

# Energy bins
print('\nEnergy bins (exposure-corrected):')
sigma_Ek=[]; sigma_Ed=[]; e_labels=[]
for lo,hi in [(0,2),(2,4),(4,8),(8,20),(20,200)]:
    me=(e_ev>=lo)&(e_ev<hi)
    if me.sum()<50: continue
    vecs_e=vecs[me]; dec_e=dec_ev[me]
    ok,_=hemi_asym(vecs_e,axis_ktf3)
    od,_=hemi_asym(vecs_e,axis_dipole)
    sce_k,sce_d=[],[]
    for _ in range(300):
        ra_s=np.random.uniform(0,360,len(dec_e))
        ls,bs=radec_to_gal(ra_s,dec_e)
        lrs=np.radians(ls);brs=np.radians(bs)
        vs=np.column_stack([np.cos(brs)*np.cos(lrs),np.cos(brs)*np.sin(lrs),np.sin(brs)])
        ak,_=hemi_asym(vs,axis_ktf3); ad,_=hemi_asym(vs,axis_dipole)
        sce_k.append(ak); sce_d.append(ad)
    sce_k=np.array(sce_k); sce_d=np.array(sce_d)
    sk=(ok-sce_k.mean())/sce_k.std()
    sd2=(od-sce_d.mean())/sce_d.std()
    sigma_Ek.append(sk); sigma_Ed.append(sd2)
    e_labels.append(f'{lo}-{hi}')
    print(f'  E={lo}-{hi} EeV (N={me.sum():,}): KTF³ σ={sk:+.2f}, Dipole σ={sd2:+.2f}')

# PLOT
fig,axes=plt.subplots(1,3,figsize=(18,6))
fig.suptitle(
    f'v84 -- Auger UHECR | REAL DATA | EXPOSURE-CORRECTED\n'
    f'KTF³ σ={sigma_k:+.2f} | Dipole σ={sigma_d:+.2f} | Sep={sep:.1f}°\n'
    f'N={len(vecs):,} | p={p_val:.3f} [{verdict}]',
    fontsize=11, fontweight='bold'
)
ax1=axes[0]
ax1.hist(dot_k,bins=30,color='#e74c3c',alpha=0.6,edgecolor='black',label=f'KTF³ σ={sigma_k:+.2f}')
ax1.hist(dot_d,bins=30,color='#3498db',alpha=0.5,edgecolor='black',label=f'Dipole σ={sigma_d:+.2f}')
ax1.axvline(0,color='black',lw=2,linestyle='--')
ax1.set_xlabel('cos(angle)'); ax1.legend(); ax1.grid(True,alpha=0.3)
ax1.set_title('Hemispheric distribution')

ax2=axes[1]
ax2.hist(sc_k,bins=30,color='#95a5a6',alpha=0.8,edgecolor='black',density=True,label='RA scrambled')
ax2.axvline(obs_k,color='red',lw=2.5,label=f'KTF³ σ={sigma_k:+.2f}')
ax2.axvline(obs_d,color='blue',lw=2,linestyle='--',label=f'Dipole σ={sigma_d:+.2f}')
ax2.set_xlabel('Asymmetry'); ax2.legend(fontsize=9); ax2.grid(True,alpha=0.3)
ax2.set_title(f'RA scrambling | p={p_val:.3f} [{verdict}]')

ax3=axes[2]
if sigma_Ek:
    x=range(len(sigma_Ek))
    ax3.bar([i-0.2 for i in x],sigma_Ek,0.35,color='#e74c3c',alpha=0.8,label='KTF³',edgecolor='black')
    ax3.bar([i+0.2 for i in x],sigma_Ed,0.35,color='#3498db',alpha=0.8,label='Dipole',edgecolor='black')
    ax3.set_xticks(list(x)); ax3.set_xticklabels(e_labels,fontsize=9)
    ax3.axhline(0,color='black'); ax3.axhline(1,color='orange',linestyle='--')
    ax3.axhline(-1,color='orange',linestyle='--'); ax3.axhline(2,color='red',linestyle=':')
ax3.set_ylabel('Sigma'); ax3.set_title('Energy-dependent'); ax3.legend(); ax3.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('v84_auger_uhecr.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'v84 FINAL: KTF³ σ={sigma_k:+.3f}, Dipole σ={sigma_d:+.3f}, Sep={sep:.1f}°, {verdict}')
print('GitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
